In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint


import matplotlib.pyplot as plt

In [2]:
qu_df = pd.read_excel("dataset/widsdatathon2025/TRAIN/TRAIN_QUANTITATIVE_METADATA.xlsx")
cat_df = pd.read_excel("dataset/widsdatathon2025/TRAIN/TRAIN_CATEGORICAL_METADATA.xlsx")
df_sol = pd.read_excel("dataset/widsdatathon2025/TRAIN/TRAINING_SOLUTIONS.xlsx")
data_dict = pd.read_excel("dataset/widsdatathon2025/Data Dictionary.xlsx")


qu_df = qu_df.drop(index=qu_df[qu_df["MRI_Track_Age_at_Scan"] < 3].index)
# qu_df = qu_df.drop(index=qu_df.query('MRI_Track_Age_at_Scan >= 15').index)
qu_df = qu_df.drop(index=qu_df[qu_df["MRI_Track_Age_at_Scan"].isna() == True].index)


cat_df = cat_df.drop(index=cat_df[cat_df["participant_id"].isin(qu_df["participant_id"].values) == False].index)
df_sol = df_sol.drop(index=df_sol[df_sol["participant_id"].isin(qu_df["participant_id"])== False].index)

qu_df = pd.merge(df_sol, qu_df, on="participant_id", how="outer")
qu_df.drop(columns=["MRI_Track_Age_at_Scan"])

cat_df = pd.merge(df_sol, cat_df, on="participant_id", how="outer")

for k in cat_df.keys():
  cat_df = cat_df.drop(index=cat_df[cat_df[k].isna() == True].index)

all_df = pd.merge(cat_df, qu_df, how="outer")

qu_df = qu_df.drop(columns=["participant_id"])
cat_df = cat_df.drop(columns=["participant_id"])
all_df = all_df.drop(columns=["participant_id"])

for k in all_df.keys():
  all_df = all_df.drop(index=all_df[all_df[k].isna() == True].index)


qu_df = (qu_df - qu_df.min())/(qu_df.max()-qu_df.min())
cat_df = (cat_df - cat_df.min())/(cat_df.max()-cat_df.min())
all_df = (all_df - all_df.min())/(all_df.max()-all_df.min())


## TTV Split

In [3]:
data = qu_df

print(data.shape)

# Classify sex only
quSexX = data.drop(columns=["Sex_F", "ADHD_Outcome"])
quSexY = data["Sex_F"]
XSex_train, XSex_test, ySex_train, ySex_test = train_test_split(quSexX, quSexY, test_size=0.2)

# Classify ADHD only
quADHDX = data.drop(columns=["Sex_F", "ADHD_Outcome"])
quADHDY = data["ADHD_Outcome"]
XADHD_train, XADHD_test, yADHD_train, yADHD_test = train_test_split(quADHDX, quADHDY, test_size=0.2)


(851, 20)


## Random Forest

In [15]:
n_estimators = 800
max_depth = 2
rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

param_dist = {'n_estimators': randint(50,500),
              'max_depth': randint(1,20)}
rand_search = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=8,cv=5)


In [16]:
# Sex Classification
rand_search.fit(XSex_train, ySex_train)
y_pred = rand_search.predict(XSex_test)
accuracy = accuracy_score(ySex_test, y_pred)
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)

Sex-Based Accuracy: 0.6023391812865497
Sex-Based Precision: 0.375
Sex-Based Recall: 0.09375


In [17]:
# ADHD Classification
rand_search.fit(XADHD_train, yADHD_train)
y_pred = rand_search.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test, y_pred)
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)

ADHD-Based Accuracy: 0.7777777777777778
ADHD-Based Precision: 0.816
ADHD-Based Recall: 0.8717948717948718


Random Forest Classifier does best when classifying ADHD only

## Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression

logReg = LogisticRegression(random_state=73)

In [8]:
# Sex Classification
logReg.fit(XSex_train, ySex_train)
y_pred = logReg.predict(XSex_test)
accuracy = accuracy_score(ySex_test, y_pred)
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)

Sex-Based Accuracy: 0.6666666666666666
Sex-Based Precision: 0.8181818181818182
Sex-Based Recall: 0.140625


In [9]:
# ADHD Classification
logReg.fit(XADHD_train, yADHD_train)
y_pred = logReg.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test, y_pred)
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)

ADHD-Based Accuracy: 0.8070175438596491
ADHD-Based Precision: 0.828125
ADHD-Based Recall: 0.905982905982906


Logistic regression appears the best out of the models tested

## SVM

In [10]:
from sklearn import svm

svmm = svm.SVC(kernel='poly') # Linear Kernel

In [11]:
# Sex Classification
svmm.fit(XSex_train, ySex_train)
y_pred = svmm.predict(XSex_test)
accuracy = accuracy_score(ySex_test, y_pred)
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)

Sex-Based Accuracy: 0.672514619883041
Sex-Based Precision: 0.7
Sex-Based Recall: 0.21875


In [12]:
# ADHD Classification
svmm.fit(XADHD_train, yADHD_train)
y_pred = svmm.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test, y_pred)
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)

ADHD-Based Accuracy: 0.7894736842105263
ADHD-Based Precision: 0.8091603053435115
ADHD-Based Recall: 0.905982905982906
